# 02 · 物理资产构建
读取模块一导入的 oven4 USD，构建真实凸碰撞体、显式物性先验及可回读的标准 USD 资产。

物性先验用于仿真和训练开发，来源、假设与接触约束随资产保存；独立实测物性和真机验证仍属于不同验收结论。全部使用当前 PhysReal-X Conda 环境。

In [ ]:
import json
import uuid

from examples._display import setup_notebook

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("physical_asset_building")

from physrealx import ArtifactRef, AssetSpec, PhysRealX

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
run_key = uuid.uuid4().hex[:12]


## 1. 参考资产

In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/real_assets_and_digital_twin/output/results.json").read_text()
)
reference_model = ArtifactRef.model_validate(upstream["reference-model.json"])
reference = json.loads(px.artifacts.get_bytes(reference_model))
print(reference["root_link"], [link["name"] for link in reference["links"]])
reference_registration = ArtifactRef.model_validate(upstream["reference-registration.json"])


## 2. 碰撞几何

In [ ]:
collision_reports = []
for link in reference["links"]:
    operation = px.assets.build_collision(
        ArtifactRef.model_validate(link["geometry"]),
        method="coacd",
        config={
            "source_unit": "m",
            "link": link["name"],
            "threshold_m": 0.003,
            "max_parts": 512,
            "max_volume_error": 0.15,
            "preprocessing": "coacd",
            "weld_tolerance_m": 0.000001,
        },
    )
    collision_reports.append(operation.output("collision_build_report"))
    print(link["name"], dict(operation.metrics))


## 3. 物性与装配

In [ ]:
# 所有未测量参数均显式列出；来源为 default，置信度为 0，不升级为实测资产。
# 关节摩擦由关节模型表达，相邻链接不重复施加接触摩擦；外部物体碰撞仍开启。
physics_prior = {
    "density_kg_m3": 650.0,
    "static_friction": 0.3,
    "dynamic_friction": 0.2,
    "restitution": 0.0,
    "joint_damping": 1.0,
    "joint_friction": 0.1,
    "effort_limit": 40.0,
    "joint_models_adjacent_contact": True,
}
assembly_input = px.artifacts.put_json(
    {
        "schema": "physrealx.reference-assembly-input/1",
        "resource_id": f"oven4.physical.{run_key}",
        "version": "1.0.0",
        "display_name": "Oven4 supplied USD with explicit physics priors",
        "reference_model": reference_model.model_dump(mode="json"),
        "registration": reference_registration.model_dump(mode="json"),
        "collision_reports": [ref.model_dump(mode="json") for ref in collision_reports],
        "physics_prior": physics_prior,
    }
)
assembly_operation = px.assets.assemble(assembly_input, method="reference_asset_assembly")
assembly = assembly_operation.output("asset_assembly")
asset = AssetSpec.model_validate(json.loads(px.artifacts.get_bytes(assembly))["asset_spec"])
print("物性证据状态:", asset.evidence_state.value)
print("各部件先验质量 kg:", {body.link: body.mass_kg.value for body in asset.physics})


## 4. USD 打包

In [ ]:
package_operation = px.assets.package(assembly, method="openusd")
usd = package_operation.output("usd_asset_bundle")
package_manifest = package_operation.output("asset_package_manifest")
print("USD 数值回读:", dict(package_operation.metrics))


## 5. 质量检查

In [ ]:
quality = px.assets.validate(asset, profile="simulation")
assert quality.passed, quality.model_dump(mode="json")
asset_ref = px.assets.register(asset)
print("可仿真资产:", asset_ref.resource_id, "关节:", len(asset.joints))
print("碰撞排除对:", asset.contact_exclusions)


## 6. 结果导出

In [ ]:
exports = {"oven4.usdz": usd, "assembly.json": assembly, "package.json": package_manifest}
results = {
    "asset": asset_ref.model_dump(mode="json"),
    "source_reference": reference_model.model_dump(mode="json"),
    "assembly": assembly.model_dump(mode="json"),
    "package": package_manifest.model_dump(mode="json"),
    "usd": usd.model_dump(mode="json"),
}
results_ref = px.artifacts.export(OUTPUT_DIR, results, files=exports)
print("结果索引:", OUTPUT_DIR / "results.json")
